<a href="https://colab.research.google.com/github/AsimHanif150/Ai-ML_internship_01-/blob/main/Assignment_1-part-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Section 0 — Setup
# ============================================================
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))


# ============================================================
# Section 1 — A rule you write by hand: stale x visible
# ============================================================
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
print(top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]])

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")


# ============================================================
# Section 2 — Let a model learn the rule — then read it
# ============================================================
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")


# ============================================================
# Section 3 — Why you can't feed the outcome back in (leakage)
# ============================================================
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))


# ============================================================
# Section 4 — Your turn (the experiment section)
# ============================================================
from sklearn.model_selection import train_test_split

# ---------------------------------------------------------------
# Experiment 1: Does depth 3/4 improve Precision@50? Still readable?
# ---------------------------------------------------------------
print("=" * 60)
print("EXPERIMENT 1: Deeper trees")
print("=" * 60)

for depth in (2, 3, 4):
    t = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42)
    t.fit(X, y)
    score = t.predict_proba(X)[:, 1]
    p50 = precision_at_k(score, y, 50)
    n_leaves = t.get_n_leaves()
    print(f"max_depth={depth}:  Precision@50 = {p50:.3f}   ({n_leaves} leaves)")

print("\nA depth-3 tree can have up to 8 leaves, depth-4 up to 16 — still readable,")
print("but each extra level roughly doubles the branching, so readability drops fast.")
print("Print one to see for yourself:")
tree_d3 = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X, y)
print(export_text(tree_d3, feature_names=features))

# ---------------------------------------------------------------
# Experiment 2: Swap features — drop impressions_90d, add engagement_rate
# ---------------------------------------------------------------
print("\n" + "=" * 60)
print("EXPERIMENT 2: Feature swap")
print("=" * 60)

features_swapped = ["content_age_days", "days_since_last_update", "avg_position",
                     "ctr", "word_count", "engagement_rate"]

missing = [f for f in features_swapped if f not in df.columns]
if missing:
    print(f"Column(s) not found, skipping: {missing}")
    print("Available columns:", df.columns.tolist())
else:
    X_swapped = df[features_swapped].replace([np.inf, -np.inf], np.nan).fillna(0)
    tree_swapped = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
    tree_swapped.fit(X_swapped, y)
    score_swapped = tree_swapped.predict_proba(X_swapped)[:, 1]
    print(f"Precision@50 (swapped features): {precision_at_k(score_swapped, y, 50):.3f}")
    print(export_text(tree_swapped, feature_names=features_swapped))
    print("\nCompare the top split here to the original tree's top split (Section 2) —")
    print("that tells you which signal the model reaches for first when impressions_90d is gone.")

# ---------------------------------------------------------------
# Experiment 3: Does the gap survive a real train/test split?
# ---------------------------------------------------------------
print("\n" + "=" * 60)
print("EXPERIMENT 3: Train/test split (not in-sample)")
print("=" * 60)

X_train, X_test, y_train, y_test, hr_train, hr_test = train_test_split(
    X, y, df["hand_rule_score"], test_size=0.3, random_state=42, stratify=y
)

tree_split = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_split.fit(X_train, y_train)
tree_test_score = tree_split.predict_proba(X_test)[:, 1]

for k in (20, 50):
    hr_p = precision_at_k(hr_test.values, y_test, k)
    tr_p = precision_at_k(tree_test_score, y_test, k)
    print(f"Precision@{k} (held-out test set):  hand rule {hr_p:.3f}   vs   tree {tr_p:.3f}")

print("\nThis tree was fit only on the training rows and scored on rows it never saw.")
print("If the gap from Section 2 shrinks or flips here, that's the tied-leaves fragility")
print("the notebook warned about — in-sample scores can flatter a low-diversity model.")

30000 pages |  declining rate: 0.542
       impressions_90d  days_since_last_update  avg_position   ctr  \
16751            61678                     194          19.7  0.15   
16514            59472                     194          24.8  0.13   
7021             25715                     194          22.2  0.23   
21268            13299                     193          10.5  0.49   
11489             7812                     194          39.0  0.01   
12045             7558                     193          17.9  0.20   
698               4590                     194          31.0  0.00   
5327              4556                     194          16.4  0.33   
26810             4429                     194          25.3  0.38   
20837             1697                     193          15.8  0.12   

      trend_direction  
16751            down  
16514            down  
7021             down  
21268            down  
11489            down  
12045            down  
698              down  
